# Save logRR for all countries and expand grid at edges

NaN values at the edge of curves are transformed to edge logRR

In [1]:
import re
import os
import time
import json
import pandas as pd
import xarray as xr
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === Path config ===
ID_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23")
# Load country names and ids
lookup_file = "country_location_ids.json"
lookup_path = os.path.join(ID_DIR, lookup_file)
with open(lookup_path, "r", encoding="utf-8") as f:
    country_names = json.load(f)  # {name: id}

In [3]:
# === Health variables ===
health_vars = ["CHRONIC_KIDNEY_DISEASE", "CARDIOMYOPATHY_MYOCARDITIS",
               "HYPERTENSIVE_HEART_DISEASE", "ISCHEMIC_HEART_DISEASE",
               "STROKE", "DIABETES", "INJ_ANIMAL", "INJ_DISASTER",
               "DROWNING", "HOMICIDE", "MECHANICAL_INJ",
               "OTHER_UNINTENTIONAL_INJ", "SUICIDE", "OTHER_TRANSPORT_INJ",
               "ROAD_TRANSPORT_INJ", "LOWER_RESPIRATORY_INFECTIONS", "COPD"]

In [4]:
def fill_edge_nans(da, dim="daily_temperature"):
    valid = da.notnull()

    # leading NaNs: no valid value anywhere before this point
    leading = valid.cumsum(dim) == 0

    # trailing NaNs: no valid value anywhere after this point
    rev = slice(None, None, -1)
    trailing = valid[{dim: rev}].cumsum(dim)[{dim: rev}] == 0

    # nearest valid value at each end
    filled = da.ffill(dim).bfill(dim)

    # use the filled values only at the edges, keep everything else as is
    return xr.where(leading | trailing, filled, da)

In [5]:
# Create file path dictionary to country specific ERFs
for health_VAR in health_vars[13:]:
    print(f"Processing {health_VAR}")

    DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23" / "RR_curves" / "temperature" / health_VAR)
    # filenames are exactly "<health_VAR>_norm_logRR_<id>.nc"
    country_files = {int(cid): DIR / f"{health_VAR}_norm_logRR_{cid}.nc" for cid in country_names.values()}

    # Check which files exist and which IDs are missing
    missing = [cid for cid, p in country_files.items() if not p.exists()]
    print("IDs in dict with no file:", missing)
    country_files = {cid: p for cid, p in country_files.items() if p.exists()}

    # Files on disk that aren't in the dict
    on_disk = {int(re.search(r"\d+", p.stem).group()) for p in DIR.glob("*.nc")
               if re.search(r"\d+", p.stem)}
    print("Files with IDs not in dict:", on_disk - set(country_files) - set(missing))  # will be 1 because of global

    # Combine per-country files into one array with a 'country' dim
    logRR = xr.concat(
        [xr.open_dataarray(p) for p in country_files.values()],
        dim=pd.Index(list(country_files.keys()), name="country"),
        join="exact",  # errors if annual_temp/daily_temp grids differ between countries
    )

    # Start the stopwatch
    start_time = time.perf_counter()

    # fill edge nans
    logRR_fill = fill_edge_nans(logRR)

    # Stop the stopwatch
    end_time = time.perf_counter()
    execution_time = end_time - start_time
    print(f"Execution time: {execution_time:.6f} seconds")

    out_file = f"{health_VAR}_norm_logRR.nc"
    out_path = os.path.join(DIR, out_file)
    print(f"Saving to {out_path}")
    logRR_fill.to_netcdf(out_path)

import resource
print(f"Peak memory: {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

print("All processing complete.")

Processing OTHER_TRANSPORT_INJ
IDs in dict with no file: []
Files with IDs not in dict: {1}
Execution time: 38.793638 seconds
Saving to /glade/work/awells/workflow/GBD23/RR_curves/temperature/OTHER_TRANSPORT_INJ/OTHER_TRANSPORT_INJ_norm_logRR.nc
Processing ROAD_TRANSPORT_INJ
IDs in dict with no file: []
Files with IDs not in dict: {1}
Execution time: 38.575156 seconds
Saving to /glade/work/awells/workflow/GBD23/RR_curves/temperature/ROAD_TRANSPORT_INJ/ROAD_TRANSPORT_INJ_norm_logRR.nc
Processing LOWER_RESPIRATORY_INFECTIONS
IDs in dict with no file: []
Files with IDs not in dict: {1}
Execution time: 38.974578 seconds
Saving to /glade/work/awells/workflow/GBD23/RR_curves/temperature/LOWER_RESPIRATORY_INFECTIONS/LOWER_RESPIRATORY_INFECTIONS_norm_logRR.nc


RuntimeError: NetCDF: HDF error

In [8]:
logRR_fill.to_netcdf(out_path)

RuntimeError: NetCDF: HDF error